In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.ecl import compute_ecl, lifetime_pd, assign_stage

portfolio = pd.read_parquet("../data/processed/portfolio_ecl.parquet")

# Remaining term in years; 12 loans lack CREDIT_TERM, so use the median
term_years = portfolio["CREDIT_TERM"].fillna(portfolio["CREDIT_TERM"].median()) / 12

portfolio["STAGE"] = assign_stage(portfolio["PD"])
portfolio["PD_LIFETIME"] = lifetime_pd(portfolio["PD"], term_years)

ecl_12m = portfolio["ECL"]
ecl_lifetime = compute_ecl(portfolio["PD_LIFETIME"], portfolio["EAD"])
portfolio["ECL_IFRS9"] = np.where(portfolio["STAGE"] == 2, ecl_lifetime, ecl_12m)

by_stage = portfolio.groupby("STAGE").agg(
    loans=("PD", "size"), ead=("EAD", "sum"), ecl=("ECL_IFRS9", "sum"),
)
by_stage["loan_share"] = by_stage["loans"] / by_stage["loans"].sum()
by_stage["coverage"] = by_stage["ecl"] / by_stage["ead"]
print(by_stage.style.format({
    "ead": "{:,.0f}", "ecl": "{:,.0f}", "loan_share": "{:.1%}", "coverage": "{:.2%}",
}).to_string())

print(f"\nTotal ECL, Stage 1 only:  {ecl_12m.sum():,.0f}")
print(f"Total ECL, with staging:  {portfolio['ECL_IFRS9'].sum():,.0f}")
print(f"Increase from staging:    {portfolio['ECL_IFRS9'].sum() / ecl_12m.sum() - 1:.1%}")

 loans ead ecl loan_share coverage
STAGE     
1 285027 173,524,528,444 4,621,746,755 92.7% 2.66%
2 22484 10,682,555,751 2,106,499,521 7.3% 19.72%


Total ECL, Stage 1 only:  5,974,216,466
Total ECL, with staging:  6,728,246,276
Increase from staging:    12.6%


## How sensitive is this to the threshold

ECL dependency on the threshold choice

In [2]:
rows = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30]:
    stage = assign_stage(portfolio["PD"], threshold=t)
    ecl = np.where(stage == 2, ecl_lifetime, ecl_12m)
    rows.append({"threshold": t, "stage2_share": (stage == 2).mean(),
                 "total_ecl": ecl.sum(), "increase": ecl.sum() / ecl_12m.sum() - 1})

sens = pd.DataFrame(rows)
print(sens.style.format({"threshold": "{:.0%}", "stage2_share": "{:.1%}",
                         "total_ecl": "{:,.0f}", "increase": "{:.1%}"}).to_string())

 threshold stage2_share total_ecl increase
0 10% 26.4% 8,210,623,141 37.4%
1 15% 13.6% 7,272,337,502 21.7%
2 20% 7.3% 6,728,246,276 12.6%
3 25% 4.0% 6,410,705,055 7.3%
4 30% 2.2% 6,225,177,473 4.2%



**Threshold sensitivity:** the SICR threshold has a large effect on the result. Moving it from 10% to 30% reduces the Stage 2 share from 26.4% to 2.2%, and total ECL from 8.2 to 6.2 billion, a difference of about a third, with PDs, LGD and exposures unchanged. The 20% threshold is a documented proxy choice; in a bank, SICR criteria combine relative and absolute PD changes with the 30-days-past-due backstop, and are justified, approved and backtested.

## Save the staged portfolio

In [3]:
portfolio.to_parquet("../data/processed/portfolio_staged.parquet", index=False)
print("Saved portfolio with stages, lifetime PD and IFRS 9 ECL")

Saved portfolio with stages, lifetime PD and IFRS 9 ECL
